In [28]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set your workspace path
WORKSPACE = r"C:\P7_pyt"
TIME_SERIES_CLUSTER_FOLDER = os.path.join(WORKSPACE, "time_series_cluster")

print(f"Working directory: {WORKSPACE}")
print(f"Time series cluster folder: {TIME_SERIES_CLUSTER_FOLDER}")

Working directory: C:\P7_pyt
Time series cluster folder: C:\P7_pyt\time_series_cluster


## Find all DBF chart files

In [30]:
# Find all DBF files (chart tables)
dbf_files = [f for f in os.listdir(TIME_SERIES_CLUSTER_FOLDER) if f.endswith('.dbf') and '_chart' in f]
dbf_files = sorted(dbf_files)

print(f"Found {len(dbf_files)} chart tables:")
for f in dbf_files:
    print(f"  - {f}")

Found 23 chart tables:
  - TSC_EMUB_chart.dbf
  - TSC_PMB_chart.dbf
  - TSC_PUB_chart.dbf
  - TSC_age_18_25_chart.dbf
  - TSC_age_26_40_chart.dbf
  - TSC_age_41_55_chart.dbf
  - TSC_age_56_69_chart.dbf
  - TSC_counts_chart.dbf
  - TSC_crime_main_y_chart.dbf
  - TSC_disp_inc_chart.dbf
  - TSC_emp_chart.dbf
  - TSC_grund_chart.dbf
  - TSC_gym_erhv_chart.dbf
  - TSC_lvu_chart.dbf
  - TSC_mean_price_chart.dbf
  - TSC_mean_sqm_chart.dbf
  - TSC_mig_in_chart.dbf
  - TSC_mig_net_chart.dbf
  - TSC_mig_out_chart.dbf
  - TSC_ool_chart.dbf
  - TSC_public_housing_chart.dbf
  - TSC_qol_chart.dbf
  - TSC_unemp_chart.dbf


## Helper function to read DBF files

In [31]:
# Try to import simpledbf, if not available, install it
try:
    from simpledbf import Dbf5
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'simpledbf'])
    from simpledbf import Dbf5

def read_dbf(dbf_path):
    """Read DBF file and return as pandas DataFrame"""
    dbf = Dbf5(dbf_path)
    df = dbf.to_dataframe()
    return df

# Test reading the first file
if dbf_files:
    test_file = os.path.join(TIME_SERIES_CLUSTER_FOLDER, dbf_files[0])
    test_df = read_dbf(test_file)
    print(f"Sample data from {dbf_files[0]}:")
    print(test_df.head())
    print(f"\nColumns: {list(test_df.columns)}")

Sample data from TSC_EMUB_chart.dbf:
   TIME_STEP  START_DATE    END_DATE  TIME_EXAG  ELEMENT  LOCATION LOC_LABEL  \
0          0  1989-01-01  1990-01-01        0.0      613       613   101_619   
1          1  1990-01-01  1991-01-01       25.6     2034       613   101_619   
2          2  1991-01-01  1992-01-01       51.2     3455       613   101_619   
3          3  1992-01-01  1993-01-01       76.8     4876       613   101_619   
4          4  1993-01-01  1994-01-01      102.4     6297       613   101_619   

   CLUST_MED  CLUST_MEAN  CLUSTER_ID  
0      51.49   53.457056           1  
1      53.21   53.568881           1  
2      52.05   53.349489           1  
3      55.52   53.493114           1  
4      54.64   53.634185           1  

Columns: ['TIME_STEP', 'START_DATE', 'END_DATE', 'TIME_EXAG', 'ELEMENT', 'LOCATION', 'LOC_LABEL', 'CLUST_MED', 'CLUST_MEAN', 'CLUSTER_ID']


## Create time series line charts for each variable

In [32]:
# Create a figure for each variable
for dbf_file in dbf_files:
    try:
        dbf_path = os.path.join(TIME_SERIES_CLUSTER_FOLDER, dbf_file)
        df = read_dbf(dbf_path)
        
        # Extract variable name from filename (e.g., 'TSC_unemp_chart.dbf' -> 'unemp')
        var_name = dbf_file.replace('TSC_', '').replace('_chart.dbf', '')
        
        # Identify time and cluster columns
        # Common column names from ArcGIS time series clustering
        time_cols = [col for col in df.columns if 'TIME' in col.upper() or 'DATE' in col.upper()]
        cluster_col = [col for col in df.columns if 'CLUSTER' in col.upper()][0] if any('CLUSTER' in col.upper() for col in df.columns) else None
        value_cols = [col for col in df.columns if col not in time_cols and cluster_col and col != cluster_col and col.upper() != 'OBJECTID']
        
        print(f"\nProcessing: {var_name}")
        print(f"  Columns: {list(df.columns)}")
        print(f"  Cluster column: {cluster_col}")
        print(f"  Data shape: {df.shape}")
        
    except Exception as e:
        print(f"Error processing {dbf_file}: {e}")


Processing: EMUB
  Columns: ['TIME_STEP', 'START_DATE', 'END_DATE', 'TIME_EXAG', 'ELEMENT', 'LOCATION', 'LOC_LABEL', 'CLUST_MED', 'CLUST_MEAN', 'CLUSTER_ID']
  Cluster column: CLUSTER_ID
  Data shape: (192, 10)

Processing: PMB
  Columns: ['TIME_STEP', 'START_DATE', 'END_DATE', 'TIME_EXAG', 'ELEMENT', 'LOCATION', 'LOC_LABEL', 'CLUST_MED', 'CLUST_MEAN', 'CLUSTER_ID']
  Cluster column: CLUSTER_ID
  Data shape: (192, 10)

Processing: PUB
  Columns: ['TIME_STEP', 'START_DATE', 'END_DATE', 'TIME_EXAG', 'ELEMENT', 'LOCATION', 'LOC_LABEL', 'CLUST_MED', 'CLUST_MEAN', 'CLUSTER_ID']
  Cluster column: CLUSTER_ID
  Data shape: (192, 10)

Processing: age_18_25
  Columns: ['TIME_STEP', 'START_DATE', 'END_DATE', 'TIME_EXAG', 'ELEMENT', 'LOCATION', 'LOC_LABEL', 'CLUST_MED', 'CLUST_MEAN', 'CLUSTER_ID']
  Cluster column: CLUSTER_ID
  Data shape: (192, 10)

Processing: age_26_40
  Columns: ['TIME_STEP', 'START_DATE', 'END_DATE', 'TIME_EXAG', 'ELEMENT', 'LOCATION', 'LOC_LABEL', 'CLUST_MED', 'CLUST_MEAN',

## Advanced: Create interactive line charts

In [33]:
def create_timeseries_chart(df, var_name, time_col=None, cluster_col=None, value_col='CLUST_MEAN'):
    """
    Create an interactive line chart showing average values per cluster over time
    Similar to the ArcGIS chart
    """
    fig = go.Figure()
    
    # If cluster_col not specified, try to find it
    if not cluster_col:
        cluster_col = [col for col in df.columns if 'CLUSTER' in col.upper()]
        cluster_col = cluster_col[0] if cluster_col else None
    
    if not time_col:
        time_col = [col for col in df.columns if 'TIME' in col.upper() or 'DATE' in col.upper()]
        time_col = time_col[0] if time_col else None
    
    # Use CLUST_MEAN as default value column
    if value_col not in df.columns:
        # Fallback to other value columns if CLUST_MEAN doesn't exist
        value_col = [col for col in df.columns if col not in [cluster_col, time_col] and 'OBJECTID' not in col.upper()]
        value_col = value_col[0] if value_col else None
    
    if not all([cluster_col, time_col, value_col]):
        print(f"Could not find required columns for {var_name}")
        print(f"  Cluster: {cluster_col}, Time: {time_col}, Value: {value_col}")
        return None
    
    # Group by cluster and plot
    df_sorted = df.sort_values(by=time_col)
    
    # Get unique clusters
    clusters = sorted(df[cluster_col].unique())
    
    # Define colors for clusters
    colors = px.colors.qualitative.Plotly
    
    for i, cluster in enumerate(clusters):
        cluster_data = df_sorted[df_sorted[cluster_col] == cluster]
        color = colors[i % len(colors)]
        
        fig.add_trace(go.Scatter(
            x=cluster_data[time_col],
            y=cluster_data[value_col],
            mode='lines+markers',
            name=f'Cluster {cluster}',
            line=dict(width=2),
            marker=dict(size=5)
        ))
    
    fig.update_layout(
        title=f"Average Time Series per Cluster - {var_name.upper()}",
        xaxis_title="Time",
        yaxis_title=f"Value: {value_col}",
        hovermode='x unified',
        height=500,
        template='plotly_white',
        font=dict(size=12)
    )
    
    return fig

# Test with first file
if dbf_files:
    dbf_path = os.path.join(TIME_SERIES_CLUSTER_FOLDER, dbf_files[0])
    df = read_dbf(dbf_path)
    var_name = dbf_files[0].replace('TSC_', '').replace('_chart.dbf', '')
    print(f"Available columns: {list(df.columns)}")
    fig = create_timeseries_chart(df, var_name)
    if fig:
        fig.show()

Available columns: ['TIME_STEP', 'START_DATE', 'END_DATE', 'TIME_EXAG', 'ELEMENT', 'LOCATION', 'LOC_LABEL', 'CLUST_MED', 'CLUST_MEAN', 'CLUSTER_ID']


## Generate all charts

In [34]:
# Create and display charts for all variables
for dbf_file in dbf_files:
    try:
        dbf_path = os.path.join(TIME_SERIES_CLUSTER_FOLDER, dbf_file)
        df = read_dbf(dbf_path)
        var_name = dbf_file.replace('TSC_', '').replace('_chart.dbf', '')
        
        fig = create_timeseries_chart(df, var_name)
        if fig:
            fig.show()
            print(f"✓ Chart created for {var_name}")
        else:
            print(f"✗ Could not create chart for {var_name}")
    
    except Exception as e:
        print(f"Error creating chart for {dbf_file}: {e}")

✓ Chart created for EMUB


✓ Chart created for PMB


✓ Chart created for PUB


✓ Chart created for age_18_25


✓ Chart created for age_26_40


...

✓ Chart created for mig_out


✓ Chart created for ool


✓ Chart created for public_housing


✓ Chart created for qol


✓ Chart created for unemp


## Export charts to HTML

In [27]:
# Create output folder for HTML charts
output_folder = os.path.join(WORKSPACE, "charts_html")
os.makedirs(output_folder, exist_ok=True)

print(f"Saving charts to: {output_folder}\n")

for dbf_file in dbf_files:
    try:
        dbf_path = os.path.join(TIME_SERIES_CLUSTER_FOLDER, dbf_file)
        df = read_dbf(dbf_path)
        var_name = dbf_file.replace('TSC_', '').replace('_chart.dbf', '')
        
        fig = create_timeseries_chart(df, var_name)
        if fig:
            output_path = os.path.join(output_folder, f"{var_name}_timeseries.html")
            fig.write_html(output_path)
            print(f"✓ Saved: {var_name}_timeseries.html")
    
    except Exception as e:
        print(f"Error saving chart for {dbf_file}: {e}")

print(f"\nAll charts saved to: {output_folder}")

Saving charts to: C:\P7_pyt\charts_html

✓ Saved: EMUB_timeseries.html
✓ Saved: EMUB_timeseries.html
✓ Saved: PMB_timeseries.html
✓ Saved: PMB_timeseries.html
✓ Saved: PUB_timeseries.html
✓ Saved: PUB_timeseries.html
✓ Saved: age_18_25_timeseries.html
✓ Saved: age_18_25_timeseries.html
✓ Saved: age_26_40_timeseries.html
✓ Saved: age_26_40_timeseries.html
✓ Saved: age_41_55_timeseries.html
✓ Saved: age_41_55_timeseries.html
✓ Saved: age_56_69_timeseries.html
✓ Saved: age_56_69_timeseries.html
✓ Saved: counts_timeseries.html
✓ Saved: crime_main_y_timeseries.html
✓ Saved: counts_timeseries.html
✓ Saved: crime_main_y_timeseries.html
✓ Saved: disp_inc_timeseries.html
✓ Saved: disp_inc_timeseries.html
✓ Saved: emp_timeseries.html
✓ Saved: emp_timeseries.html
✓ Saved: grund_timeseries.html
✓ Saved: grund_timeseries.html
✓ Saved: gym_erhv_timeseries.html
✓ Saved: gym_erhv_timeseries.html
✓ Saved: lvu_timeseries.html
✓ Saved: mean_price_timeseries.html
✓ Saved: lvu_timeseries.html
✓ Saved: mean